# Option Pricing Model (OPM) - Project Horizon Full Valuation vGameTheory

This notebook is a separate game-theory extension of the Project Horizon FULL valuation. It preserves the same contract inputs and market assumptions, but changes the overlap logic from a neutral same-bucket approximation to a discrete-time zero-sum game approximation from Olympus BV's perspective.

## Valuation objective
- Primary output: integrated net contract fair value under a game-theory approximation, not separate call and put values priced independently and then netted
- DLOM: retained in the primary valuation
- Regulatory lapse risk: discussed qualitatively and not quantified in the primary fair value
- Clean-value convention: leakage, dividends, and SPA claim adjustments remain outside the primary model unless separately modeled

## Contract features captured in this build
- Call exercise window: after Year 3 to Year 5
- Put exercise window: after Year 3.5 to Year 6
- One Live Notice: overlap interaction between call and put rights
- Partial and multiple call exercise: where contractually available
- Residual-share put logic: Titan's put applies only to shares still held at the exercise date
- Strike mechanics: SPA price plus Applicable Coupon accrued to option completion

## Explicit modeling assumptions
- Windowed-American rights are approximated on a monthly Bermudan grid
- Base case assumes partial call exercise is available
- Overlap region is approximated as a discrete-time zero-sum game from Olympus BV's perspective
- The remaining Titan stake is represented on a finer 1% state grid from 25% down to 7%, plus 0%
- Completion lag is set to zero in the primary valuation as a simplifying assumption: for valuation purposes, option exercise notice and option completion are assumed to occur on the same date
- Slow sensitivity cells are gated by a notebook flag so they can be skipped during normal runs

## Build notes
- This notebook is closer to a stochastic game than the prior FULL notebook, but it is still a numerical approximation rather than a closed-form continuous-time solution
- The overlap region is treated as Olympus maximizing value and Titan minimizing Olympus value, subject to the contract's call and put constraints
- The exact share-level optimum is still approximated on a 1% grid rather than solved over every individual share count
- The original FULL notebook is preserved separately and should be read as the earlier non-game-theoretic build
- Any row still marked as pending in the source register below should be treated as not yet fully cleared for external circulation

## 1. Inputs

### Step 1A. Input Provenance and Citation Control

Before any number is relied on externally, each input should be tagged as one of three types: contract-derived, SPA or transaction-document-derived, or external market data. For client, adviser, auditor, or bank circulation, each row should carry an exact citation.

If an SPA-derived row does not yet show the exact SPA paragraph reference, that row should be treated as still open.

| Input | Base value | Type | Source / citation requirement | Status |
|---|---|---|---|---|
| `TOTAL_EQUITY_USD` / `SPA_price` | USD 3.2bn / USD 577.19 per share | SPA-derived | Exact SPA paragraph citation required. The SPA file is not present in the current workspace, so this citation remains outstanding. | Pending |
| `INITIAL_TITAN_PCT` / `OPTION_SHARES` | 25% / 1,386,020 shares | Contract-derived | Option Agreement Schedule 3 definition of `Option Shares`; Titan warranty at Clause 13.2.2 | Complete |
| `CALL_COUPONS` / `PUT_COUPONS` | Per code below | Contract-derived | Option Agreement Schedule 3 definition of `Applicable Coupon` | Complete |
| `call_start`, `call_end`, `put_start`, `put_end` | 3.0y-5.0y / 3.5y-6.0y | Contract-derived | Clause 4.1 and Schedule 3 definitions of `Call Option Period`, `Put Option Period`, and `Overlap Period` | Complete |
| `MIN_CALL_TRANCHE_PCT` / `RESIDUAL_FLOOR_PCT` | 5% / 7% | Contract-derived | Clause 2.2 minimum-tranche and residual-floor restrictions | Complete |
| `COMPLETION_LAG_YEARS` | 0.0 years | Modeling assumption | Schedule 3 definition of `Option Completion Date` contemplates 20 business days; primary valuation sets lag to zero as an explicit simplifying assumption | Complete as assumption |
| `OLYMPUS_PRIORITY_IN_OVERLAP` | 50% | Modeling assumption | Clause 8.1 gives earlier notice priority; 50/50 is the model's neutral same-bucket approximation | Complete as assumption |
| `asset_volatility` | 20.0% | External market data / judgment | Selected KO bottler peer asset-volatility analysis, Capital IQ, as of 31-December-2025 | Complete |
| `risk_free_rate` | 4.929% | External market data | Bloomberg USD risk-free input, as of 31-December-2025 | Complete |

In [4]:
import numpy as np

# ============================================================
# INPUT PROVENANCE / CITATION CONTROL
# ============================================================
# External-circulation rule:
# - contract-derived inputs cite the signed Option Agreement by clause / schedule;
# - external market inputs cite provider and date;
# - SPA-derived inputs cite the exact SPA paragraph before issue.
INPUT_SOURCE_REGISTER = {
    "TOTAL_EQUITY_USD / SPA_price": {
        "citation": "SPA-derived transaction input. Exact SPA paragraph citation remains required before external circulation; the SPA file is not present in the current workspace.",
        "status": "pending",
    },
    "INITIAL_TITAN_PCT / OPTION_SHARES": {
        "citation": "Option Agreement Schedule 3 definition of 'Option Shares'; Titan warranty at Clause 13.2.2.",
        "status": "complete",
    },
    "CALL_COUPONS / PUT_COUPONS": {
        "citation": "Option Agreement Schedule 3 definition of 'Applicable Coupon'.",
        "status": "complete",
    },
    "call_start / call_end / put_start / put_end": {
        "citation": "Clause 4.1 together with Schedule 3 definitions of 'Call Option Period', 'Put Option Period' and 'Overlap Period'.",
        "status": "complete",
    },
    "MIN_CALL_TRANCHE_PCT / RESIDUAL_FLOOR_PCT": {
        "citation": "Clause 2.2 minimum-tranche and residual-floor restrictions.",
        "status": "complete",
    },
    "COMPLETION_LAG_YEARS": {
        "citation": "Schedule 3 definition of 'Option Completion Date'; primary valuation sets lag to zero as an explicit simplifying assumption.",
        "status": "complete",
    },
    "GAME_TIE_PRIORITY_TO_OLYMPUS": {
        "citation": "Clause 8.1 gives earlier notice priority; same-bucket exact ordering is not observable on a monthly grid, so this notebook uses an explicit tie-priority approximation.",
        "status": "complete",
    },
    "asset_volatility": {
        "citation": "Selected KO bottler peer asset-volatility analysis, Capital IQ, as of 31-Dec-2025.",
        "status": "complete",
    },
    "risk_free_rate": {
        "citation": "Bloomberg USD risk-free input, as of 31-Dec-2025.",
        "status": "complete",
    },
}

# ============================================================
# CONTRACT TERMS / PRIMARY MODELING ASSUMPTIONS
# ============================================================
# SPA-derived reference equity value.
# Source required before external issue: exact SPA paragraph citation.
TOTAL_EQUITY_USD = 3_200_000_000

# Source: Option Agreement Schedule 3 definition of "Option Shares";
# corroborated by Titan warranty at Clause 13.2.2.
INITIAL_TITAN_PCT = 25
OPTION_SHARES = 1_386_020
TOTAL_SHARES = OPTION_SHARES / (INITIAL_TITAN_PCT / 100)
SPA_price = TOTAL_EQUITY_USD / TOTAL_SHARES

# Source: Option Agreement Schedule 3 definition of "Applicable Coupon".
CALL_COUPONS = np.array([0.0275, 0.0300, 0.0300, 0.0420, 0.0420], dtype=float)
PUT_COUPONS = np.array([0.0275, 0.0300, 0.0300, 0.0410, 0.0410, 0.0410], dtype=float)

# Source: Clause 4.1 together with Schedule 3 definitions of
# "Call Option Period", "Put Option Period" and "Overlap Period".
call_start, call_end = 3.0, 5.0
put_start, put_end = 3.5, 6.0

# Source: Clause 2.2 minimum-tranche and residual-floor restrictions.
MIN_CALL_TRANCHE_PCT = 5
RESIDUAL_FLOOR_PCT = 7
STATE_PCTS = np.array([0] + list(range(7, 26)), dtype=int)

# Contract reference: Schedule 3 definition of "Option Completion Date".
# Primary simplifying assumption for this client-facing build:
# notice date and option completion date are aligned for valuation purposes.
COMPLETION_LAG_YEARS = 0.0

# Working base case: partial call exercise is available.
ALLOW_PARTIAL_CALLS = True

# Same-bucket game tie-break approximation within the monthly grid.
GAME_TIE_PRIORITY_TO_OLYMPUS = 0.50

# Notebook runtime control: slow sensitivity cells are skipped unless enabled.
RUN_SLOW_SENSITIVITIES = False

# ============================================================
# MARKET / ECONOMIC ASSUMPTIONS
# ============================================================
# Source: selected KO bottler peer asset-volatility analysis,
# Capital IQ, as of 31-Dec-2025, rounded to 20.0%.
asset_volatility = 0.20

# Source: Bloomberg USD risk-free input, as of 31-Dec-2025.
risk_free_rate = 0.04929

# ============================================================
# GRID / SIMULATION SETTINGS
# ============================================================
# Core vGameTheory valuation uses a deterministic monthly stock tree.
steps_per_year = 12
dt = 1 / steps_per_year
total_years = 6
tree_steps = int(total_years / dt)
total_steps = tree_steps + 1
time_grid = np.arange(total_steps) * dt

# Simulated paths are retained for DLOM and optional diagnostics.
simulations = 1_000_000

pending_source_items = [
    name for name, meta in INPUT_SOURCE_REGISTER.items() if meta["status"] != "complete"
]

np.random.seed(42)
print(f"SPA price per share:                      USD {SPA_price:,.2f}")
print(f"Initial option block value:               USD {SPA_price * OPTION_SHARES:,.0f}")
print(f"Base remaining state:                     {INITIAL_TITAN_PCT}% of total share capital")
print("Completion lag assumption:                0.000 years")
print("                                           Simplifying assumption: notice date = completion date")
print(f"Partial call assumption:                  {'Enabled' if ALLOW_PARTIAL_CALLS else 'Disabled'}")
print(f"Same-bucket tie priority to Olympus:      {GAME_TIE_PRIORITY_TO_OLYMPUS:.0%}")
print(f"State grid used in vGameTheory:           {STATE_PCTS[1]}% to {STATE_PCTS[-1]}% plus 0%")
print(f"Slow sensitivity cells enabled:           {'Yes' if RUN_SLOW_SENSITIVITIES else 'No - skipped by default'}")
print(f"Asset volatility:                         {asset_volatility:.2%}  (Capital IQ, 31-Dec-2025)")
print(f"Risk-free rate:                           {risk_free_rate:.3%}  (Bloomberg, 31-Dec-2025)")
print(f"MC paths retained for DLOM only:          {simulations:,}")
if pending_source_items:
    print("Source-control exceptions:")
    for name in pending_source_items:
        print(f"- {name}: {INPUT_SOURCE_REGISTER[name]['citation']}")
else:
    print("Source-control exceptions:                None")

SPA price per share:                      USD 577.19
Initial option block value:               USD 800,000,000
Base remaining state:                     25% of total share capital
Completion lag assumption:                0.000 years
                                           Simplifying assumption: notice date = completion date
Partial call assumption:                  Enabled
Same-bucket tie priority to Olympus:      50%
State grid used in vGameTheory:           7% to 25% plus 0%
Slow sensitivity cells enabled:           No - skipped by default
Asset volatility:                         20.00%  (Capital IQ, 31-Dec-2025)
Risk-free rate:                           4.929%  (Bloomberg, 31-Dec-2025)
MC paths retained for DLOM only:          1,000,000
Source-control exceptions:
- TOTAL_EQUITY_USD / SPA_price: SPA-derived transaction input. Exact SPA paragraph citation remains required before external circulation; the SPA file is not present in the current workspace.


## 2. Contract Strike Paths

The agreement defines purchase price by reference to the SPA price plus the Applicable Coupon accrued to option completion.

The key point on completion lag is the following:
- It is the gap between **option exercise notice** and **option completion** under the Option Agreement.
- It is **not** the gap between the valuation date and the original SPA close.
- It is **not** an assumption that the transaction or PPA remained unfinished after the valuation date.

In principle, if option completion occurs after notice, the coupon continues to accrue through that completion date. In this client-facing build, that lag is set to **zero** as a practical simplifying assumption. That means the model assumes, for valuation purposes, that exercise and completion occur on the same date, so no additional coupon accrues after exercise notice in the primary valuation.

In [5]:
def accrued_price(base_price, coupons, t_years):
    """Accrue SPA price by the contractual Applicable Coupon schedule."""
    acc = base_price
    remaining = max(float(t_years), 0.0)
    for rate in coupons:
        if remaining <= 0:
            break
        year_slice = min(1.0, remaining)
        acc *= (1 + rate) ** year_slice
        remaining -= year_slice
    if remaining > 0:
        acc *= (1 + coupons[-1]) ** remaining
    return acc


def build_notice_strike_path(base_price, coupons, time_grid, completion_lag_years, rf):
    """
    Build strike paths to expected option completion.

    With completion_lag_years = 0.0, this simplifies to notice date = completion date.
    """
    strike_completion = np.zeros_like(time_grid)
    strike_pv = np.zeros_like(time_grid)
    completion_grid = np.zeros_like(time_grid)

    for idx, t_notice in enumerate(time_grid):
        t_completion = t_notice + completion_lag_years
        strike_completion[idx] = accrued_price(base_price, coupons, t_completion)
        strike_pv[idx] = np.exp(-rf * completion_lag_years) * strike_completion[idx]
        completion_grid[idx] = t_completion

    return strike_completion, strike_pv, completion_grid


call_strike_completion, call_strike_pv, call_completion_grid = build_notice_strike_path(
    SPA_price, CALL_COUPONS, time_grid, COMPLETION_LAG_YEARS, risk_free_rate
)
put_strike_completion, put_strike_pv, put_completion_grid = build_notice_strike_path(
    SPA_price, PUT_COUPONS, time_grid, COMPLETION_LAG_YEARS, risk_free_rate
)

for yr in [3.0, 3.5, 5.0, 6.0]:
    idx = int(round(yr / dt))
    if idx < total_steps:
        print(
            f"Exercise at Year {yr:>3.1f} | "
            f"Call strike used: {call_strike_completion[idx]:.4f} | "
            f"Put strike used: {put_strike_completion[idx]:.4f}"
        )

Exercise at Year 3.0 | Call strike used: 629.1827 | Put strike used: 629.1827
Exercise at Year 3.5 | Call strike used: 642.2596 | Put strike used: 641.9514
Exercise at Year 5.0 | Call strike used: 683.1439 | Put strike used: 681.8333
Exercise at Year 6.0 | Call strike used: 711.8360 | Put strike used: 709.7885


## 3. Simulate Equity Value Paths

Risk-neutral GBM paths are still simulated in this notebook, but in the vGameTheory build they are retained primarily for DLOM and optional diagnostics.

The core game-theory contract value below is produced by discrete-time backward induction on a recombining stock-price tree rather than by least-squares Monte Carlo.

In [6]:
def simulate_gbm_paths(n_paths, total_steps, dt, spot0, rf, vol, seed=42):
    rng = np.random.default_rng(seed)
    paths = np.zeros((n_paths, total_steps), dtype=float)
    paths[:, 0] = spot0

    drift = (rf - 0.5 * vol**2) * dt
    diffusion = vol * np.sqrt(dt)

    for step in range(1, total_steps):
        z = rng.standard_normal(n_paths)
        paths[:, step] = paths[:, step - 1] * np.exp(drift + diffusion * z)

    return paths


paths = simulate_gbm_paths(
    n_paths=simulations,
    total_steps=total_steps,
    dt=dt,
    spot0=SPA_price,
    rf=risk_free_rate,
    vol=asset_volatility,
    seed=42,
)

print(f"Simulated {simulations:,} paths x {total_steps} steps.")
print(
    f"Mean equity at Year 6: {paths[:, -1].mean():.2f} "
    f"(risk-neutral expectation ~ {SPA_price * np.exp(risk_free_rate * total_years):.2f})"
)

Simulated 1,000,000 paths x 73 steps.
Mean equity at Year 6: 776.08 (risk-neutral expectation ~ 775.82)


## 4. Integrated Contract Engine

The vGameTheory build values the Option Agreement as a linked two-party stopping game from Olympus BV's perspective rather than as independent call and put legs.

### High-level methodological structure

- Time grid: monthly Bermudan exercise grid
- Stock process: recombining risk-neutral tree
- Ownership state grid: Titan's remaining shareholding is represented on a finer 1% grid from 25% down to 7%, plus 0%
- Olympus action set: wait or call any contract-feasible tranche on that grid
- Titan action set: wait or, when live, put all remaining shares in the current state
- Overlap treatment: in the overlap window the monthly decision is treated as a zero-sum stage game, with Olympus maximizing and Titan minimizing Olympus's value
- Numerical technique: backward induction on the stock tree and remaining-state grid

### What the remaining-state grid means

- The contract starts today in the 25% state. That is why the 25% line later in the notebook equals the base-case contract value.
- The 24% down to 7% states are conditional states the model could move into if earlier call exercises had already reduced Titan's holding.
- The 0% state means the contract has been exhausted because Titan no longer holds any shares under the agreement.
- The remaining-state table later in the notebook is therefore a diagnostic by state, not a probability-weighted average across states.

### How partial exercise amount is chosen in this build

- If Titan currently holds `c%`, Olympus may choose any contract-feasible call size `x%` such that `x >= 5` and Titan is left with either `c - x = 0` or `c - x >= 7`.
- The model therefore does not hard-code "call 5%" versus "call 15%". It evaluates the admissible call-size menu on the finer 1% state grid.
- Olympus then picks the action that maximizes value after taking Titan's best response into account in the overlap period.
- That is the key rationale for partial exercise here: a partial call can reduce worst-case residual put exposure while preserving value on the shares not yet acquired.

### Why this is a game-theory approximation

- From Year 3.0 to 3.5 only Olympus can act, so the problem is single-sided in that region.
- From Year 3.5 to 5.0 both Olympus and Titan can act, so the problem becomes strategic.
- From Year 5.0 to 6.0 only Titan can act, so the problem again becomes single-sided, but now in Titan's favour.
- In the overlap window, the model treats the contract as a zero-sum game from Olympus BV's perspective: Olympus seeks to maximize contract value and Titan seeks to minimize it.

### Same-bucket tie handling

- Clause 8.1 gives priority to the earlier notice, but a monthly grid does not observe exact intra-month ordering.
- "Both sides would act in the same modeled month" does **not** mean both notices legally complete together.
- It means that, at that monthly node, Olympus's preferred move is "call now" rather than wait and Titan's preferred move is "put now" rather than wait.
- Those incentives can overlap even in a zero-sum contract because each side may want to **pre-empt** the other rather than let the other side move first.
- The notebook therefore uses an explicit same-bucket tie-priority parameter, `GAME_TIE_PRIORITY_TO_OLYMPUS`, to approximate who prevails when both sides prefer immediate action in that same modeled month.
- If Olympus is given priority in a same-bucket clash, the model assumes Olympus first acquires its chosen call tranche and Titan can then only put the residual holding.
- If Titan is given priority, Titan's put on the full current residual block prevails and Olympus's same-bucket call is effectively crowded out.

### Put-holder value versus call-holder value

- Under the same pricing assumptions, the contract is zero-sum between the two counterparties.
- So the fair value to Titan is the negative of the fair value to Olympus, provided both are using the same model, same discounting, same volatility, and no party-specific funding, credit, tax, or liquidity adjustments.
- Economically that is consistent with Titan holding the long put / short call side of the same contract that Olympus values as long call / short put.
- Opposite contract value does **not** mean opposite exercise timing at every node; both sides can still prefer to act in the same month if each is trying to get there first.

### What the current build is and is not

- It is a discrete-time game-theory approximation of the linked contract.
- It is not a standalone call valuation minus a standalone put valuation.
- It is not a full continuous-time closed-form Dynkin solution.
- It still approximates the exact feasible share count on a tractable monthly time grid and 1% remaining-state grid.
- It is a materially richer strategic model than the earlier coarse same-bucket approximation.

In [7]:
def shares_for_pct(pct):
    return TOTAL_SHARES * (pct / 100.0)


def allowed_call_targets(current_pct, allow_partial_calls, state_pcts=None):
    state_space = STATE_PCTS if state_pcts is None else np.array(state_pcts, dtype=int)
    if current_pct == 0:
        return []
    if not allow_partial_calls:
        return [0]
    targets = [
        int(target)
        for target in state_space
        if target < current_pct
        and (current_pct - target) >= MIN_CALL_TRANCHE_PCT
        and (target == 0 or target >= RESIDUAL_FLOOR_PCT)
    ]
    return sorted(targets, reverse=True)


def summarize_call_menu(current_pct, allow_partial_calls):
    targets = allowed_call_targets(current_pct, allow_partial_calls)
    if not targets:
        return "no admissible calls"
    partial_targets = [target for target in targets if target != 0]
    parts = []
    if partial_targets:
        call_sizes = sorted(current_pct - target for target in partial_targets)
        remaining_states = sorted(partial_targets)
        parts.append(
            f"partial calls {call_sizes[0]}%-{call_sizes[-1]}% in 1% steps -> leave {remaining_states[0]}%-{remaining_states[-1]}% remaining"
        )
    if 0 in targets:
        parts.append(f"call {current_pct}% -> 0% remaining")
    return "; ".join(parts)


def build_recombining_stock_tree(spot0, rf, vol, dt, n_steps):
    up = np.exp(vol * np.sqrt(dt))
    down = 1.0 / up
    growth = np.exp(rf * dt)
    up_prob = (growth - down) / (up - down)
    if up_prob <= 0 or up_prob >= 1:
        raise ValueError("Risk-neutral probability is outside (0, 1); tree parameters are invalid.")
    stock_tree = []
    for step in range(n_steps + 1):
        up_moves = np.arange(step + 1)
        down_moves = step - up_moves
        stock_tree.append(spot0 * (up ** up_moves) * (down ** down_moves))
    return stock_tree, up_prob, np.exp(-rf * dt)


def tree_continuation(next_values, discount, up_prob):
    return discount * ((1.0 - up_prob) * next_values[:-1] + up_prob * next_values[1:])


def in_window(t, start, end, tol=1e-10):
    return (t + tol) >= start and (t - tol) <= end


def value_game_contract_tree(
    time_grid,
    dt,
    rf,
    spot0,
    vol,
    call_strike_pv,
    put_strike_pv,
    allow_partial_calls,
    tie_priority_to_olympus,
    state_pcts,
    initial_titan_pct,
    option_shares,
):
    n_steps = len(time_grid) - 1
    stock_tree, up_prob, discount = build_recombining_stock_tree(spot0, rf, vol, dt, n_steps)
    positive_states = [int(pct) for pct in state_pcts if int(pct) > 0]
    next_values_by_state = None

    for step in range(n_steps, -1, -1):
        t = time_grid[step]
        node_count = step + 1
        current_spot = stock_tree[step]

        if next_values_by_state is None:
            continuation_by_state = {pct: np.zeros(node_count, dtype=float) for pct in state_pcts}
        else:
            continuation_by_state = {
                pct: tree_continuation(next_values_by_state[pct], discount, up_prob)
                for pct in state_pcts
            }

        call_live = in_window(t, call_start, call_end)
        put_live = in_window(t, put_start, put_end)
        call_intrinsic = np.maximum(current_spot - call_strike_pv[step], 0.0) if call_live else np.zeros(node_count, dtype=float)
        put_intrinsic = np.maximum(put_strike_pv[step] - current_spot, 0.0) if put_live else np.zeros(node_count, dtype=float)

        current_values_by_state = {0: np.zeros(node_count, dtype=float)}

        for current_pct in positive_states:
            wait_value = continuation_by_state[current_pct]

            if call_live and not put_live:
                state_value = wait_value.copy()
                for target_pct in allowed_call_targets(current_pct, allow_partial_calls, state_pcts):
                    called_shares = shares_for_pct(current_pct - target_pct)
                    call_value = call_intrinsic * called_shares + continuation_by_state[target_pct]
                    state_value = np.maximum(state_value, call_value)

            elif put_live and not call_live:
                put_value = -(put_intrinsic * shares_for_pct(current_pct))
                state_value = np.minimum(wait_value, put_value)

            elif call_live and put_live:
                put_value = -(put_intrinsic * shares_for_pct(current_pct))
                state_value = np.minimum(wait_value, put_value)
                for target_pct in allowed_call_targets(current_pct, allow_partial_calls, state_pcts):
                    called_shares = shares_for_pct(current_pct - target_pct)
                    call_immediate = call_intrinsic * called_shares
                    call_then_wait = call_immediate + continuation_by_state[target_pct]
                    olympus_first_then_titan_puts_residual = call_immediate - (put_intrinsic * shares_for_pct(target_pct))
                    simultaneous_value = (
                        tie_priority_to_olympus * olympus_first_then_titan_puts_residual
                        + (1.0 - tie_priority_to_olympus) * put_value
                    )
                    titan_best_response = np.minimum(call_then_wait, simultaneous_value)
                    state_value = np.maximum(state_value, titan_best_response)

            else:
                state_value = wait_value

            current_values_by_state[current_pct] = state_value

        next_values_by_state = current_values_by_state

    initial_paths = np.array([next_values_by_state[initial_titan_pct][0]], dtype=float)
    state_values_total = {pct: next_values_by_state[pct][0] for pct in next_values_by_state}
    value_total = float(initial_paths[0])
    return {
        "value_paths_total": initial_paths,
        "value_per_share": value_total / option_shares,
        "value_total": value_total,
        "value_std_total": 0.0,
        "value_se_total": 0.0,
        "state_values_total": state_values_total,
        "tree_up_probability": up_prob,
        "tree_discount": discount,
    }


def run_vgame_model_scenario(vol, rf, allow_partial_calls, tie_priority_to_olympus):
    _, scenario_call_pv, _ = build_notice_strike_path(
        SPA_price, CALL_COUPONS, time_grid, COMPLETION_LAG_YEARS, rf
    )
    _, scenario_put_pv, _ = build_notice_strike_path(
        SPA_price, PUT_COUPONS, time_grid, COMPLETION_LAG_YEARS, rf
    )
    return value_game_contract_tree(
        time_grid=time_grid,
        dt=dt,
        rf=rf,
        spot0=SPA_price,
        vol=vol,
        call_strike_pv=scenario_call_pv,
        put_strike_pv=scenario_put_pv,
        allow_partial_calls=allow_partial_calls,
        tie_priority_to_olympus=tie_priority_to_olympus,
        state_pcts=STATE_PCTS,
        initial_titan_pct=INITIAL_TITAN_PCT,
        option_shares=OPTION_SHARES,
    )

### 4A. How Exercise Is Applied Inside the Game-Theory Approximation

This is the key point: the amount of call exercise is chosen **inside** the backward-induction game, not after the valuation as a separate overlay.

For each monthly decision date, for each stock-price node, and for each current remaining-state, the vGameTheory build does the following:

1. Compute the value of **waiting**.
2. If only the call is live, let Olympus choose the admissible call size that maximizes value.
3. If only the put is live, let Titan choose whether to wait or put all remaining shares so as to minimize Olympus's value.
4. If both are live, solve a monthly **max/min stage game**: Olympus chooses the call action, Titan chooses the response, and the notebook values the resulting outcome from Olympus BV's perspective.

For example, from the **25%** state in the overlap region, Olympus can call any contract-feasible amount from **5%** to **18%** in 1% increments, or call the full **25%** to zero.

That means the model is no longer asking only "exercise or wait?" It is asking a richer question:

- if Olympus acts now, **how much** should Olympus call; and
- given that Olympus choice, what would Titan rationally do if Titan is also trying to maximize its own value, equivalently minimize Olympus's value?

### Why partial exercise can be optimal

The economic trade-off is the following:

- a larger call acquires more shares immediately and therefore captures more upside at once; but
- a larger call also gives up more timing optionality on the called tranche; and
- in the overlap region, the residual block left behind is what Titan can still try to put if Titan acts strategically.

So if Olympus calls **5%** rather than **15%**, that is not arbitrary. It means the model has judged that, after taking Titan's best response into account, the smaller reduction in put exposure plus the preserved optionality on the residual block is worth more than a larger immediate call.

If instead **15%** is better, the notebook should choose **15%**. The amount is meant to be endogenous to the game-theory approximation, not a fixed narration choice.

### Same-bucket interpretation

On a monthly grid, exact day-of-month notice order is not observed. So when the notebook says that both sides would act in the same modeled month, it does **not** mean both notices legally complete at once.

It means something narrower: at that monthly node, Olympus prefers **call now** to waiting, and Titan also prefers **put now** to waiting. That is a collision of exercise incentives on the coarse monthly grid, not simultaneous legal completion.

That can happen even though the contract is zero-sum. Zero-sum means the value gained by one side is lost by the other. It does **not** mean the two sides cannot both want to move immediately in order to get there first.

The `GAME_TIE_PRIORITY_TO_OLYMPUS` parameter is therefore just a tractable way of approximating which notice lands first inside that month when both sides prefer immediate action on the monthly grid.

### Put-holder perspective

Under the same pricing assumptions, the put-holder side is the mirror image of the call-holder side on this contract.

- Olympus BV is valuing the instrument as **long call / short put**.
- Titan is valuing the same instrument as **long put / short call**.
- Therefore, absent party-specific adjustments, the value to Titan is the **negative** of the value to Olympus.

So if the contract is worth **+USD 100M** to Olympus under one set of assumptions, it is worth **-USD 100M** to Titan under that same set of assumptions, and vice versa.

That symmetry breaks only if the two sides are being valued with different assumptions, for example different funding, credit, tax, or liquidity adjustments.

## 5. Base Integrated Valuation

Run the integrated contract engine from the initial 25% state and retain the pathwise discounted values for reporting, DLOM, and the fixed-policy precision summary.

In [8]:
full_result = value_game_contract_tree(
    time_grid=time_grid,
    dt=dt,
    rf=risk_free_rate,
    spot0=SPA_price,
    vol=asset_volatility,
    call_strike_pv=call_strike_pv,
    put_strike_pv=put_strike_pv,
    allow_partial_calls=ALLOW_PARTIAL_CALLS,
    tie_priority_to_olympus=GAME_TIE_PRIORITY_TO_OLYMPUS,
    state_pcts=STATE_PCTS,
    initial_titan_pct=INITIAL_TITAN_PCT,
    option_shares=OPTION_SHARES,
)

full_value_paths_total = full_result["value_paths_total"]
full_value_total = full_result["value_total"]
full_value_per_share = full_result["value_per_share"]
full_value_std_total = full_result["value_std_total"]
full_value_se_total = full_result["value_se_total"]
state_values_total = full_result["state_values_total"]
titan_value_total = -full_value_total
titan_value_per_share = -full_value_per_share

print(f"vGameTheory contract value to Olympus (per initial option share): USD {full_value_per_share:,.4f}")
print(f"vGameTheory contract value to Olympus (total):                   USD {full_value_total / 1e6:,.1f}M")
print(f"Mirror value to Titan under same assumptions:                   USD {titan_value_total / 1e6:,.1f}M")
print("Core contract value uses deterministic backward induction, so no Monte Carlo standard error applies here.")

vGameTheory contract value to Olympus (per initial option share): USD 101.6842
vGameTheory contract value to Olympus (total):                   USD 140.9M
Mirror value to Titan under same assumptions:                   USD -140.9M
Core contract value uses deterministic backward induction, so no Monte Carlo standard error applies here.


## 6. Summary Before DLOM

This is the clean integrated contract value from Olympus BV's perspective before any marketability haircut under the vGameTheory approximation.

The remaining-state table below should be read as a diagnostic of separate conditional states:
- 25% = the current initial state at the valuation date
- lower remaining percentages = conditional reduced states after hypothetical prior call exercise
- 0% = exhausted contract

Because the core vGameTheory valuation is produced by deterministic backward induction on a stock tree, there is no Monte Carlo confidence interval for the pre-DLOM contract value itself.

In [9]:
print("=" * 72)
print("vGameTheory contract valuation before DLOM")
print("=" * 72)
print(f"Value to Olympus:               USD {full_value_total / 1e6:>8.1f}M")
print(f"Value to Olympus per share:     USD {full_value_per_share:>8.2f}")
print(f"Mirror value to Titan:          USD {titan_value_total / 1e6:>8.1f}M")
print(f"Mirror value to Titan/share:    USD {titan_value_per_share:>8.2f}")
print("Core contract value is deterministic on the stock tree; no MC CI applies before DLOM.")
print("-" * 72)
print("Action menu represented inside the vGameTheory engine")
print("-" * 72)
for current_pct in [25, 20, 15, 10]:
    print(
        f"State {current_pct:>2}% | {summarize_call_menu(current_pct, ALLOW_PARTIAL_CALLS)} | "
        f"if put is live: put all remaining {current_pct}%"
    )
print("State  0% | contract exhausted")
print("-" * 72)
print("Financial interpretation of partial call choice")
print("- Olympus chooses the call amount that maximizes value after Titan's best response is considered")
print("- Titan's best response in overlap is to minimize Olympus value, equivalently maximize Titan value")
print("- that is why the amount of partial call is endogenous in this notebook rather than narrated externally")
print("- partial exercise can be optimal because it reduces residual put exposure without surrendering all optionality at once")
print("- under shared assumptions, Titan's contract value is the negative of Olympus's contract value")
print("-" * 72)
print("Conditional contract values by Titan remaining-state")
print("Diagnostic only; not a probability-weighted average across states")
print("-" * 72)
for pct in [25, 20, 15, 10, 7, 0]:
    total_value = state_values_total[pct]
    if pct > 0:
        per_remaining_share = total_value / shares_for_pct(pct)
        state_role = "current initial state" if pct == INITIAL_TITAN_PCT else "conditional reduced state"
        print(
            f"Titan remaining {pct:>2}% | "
            f"{state_role:<25} | "
            f"contract value {total_value / 1e6:>8.1f}M | "
            f"per remaining share {per_remaining_share:>8.2f}"
        )
    else:
        print("Titan remaining  0% | contract exhausted")

vGameTheory contract valuation before DLOM
Value to Olympus:               USD    140.9M
Value to Olympus per share:     USD   101.68
Mirror value to Titan:          USD   -140.9M
Mirror value to Titan/share:    USD  -101.68
Core contract value is deterministic on the stock tree; no MC CI applies before DLOM.
------------------------------------------------------------------------
Action menu represented inside the vGameTheory engine
------------------------------------------------------------------------
State 25% | partial calls 5%-18% in 1% steps -> leave 7%-20% remaining; call 25% -> 0% remaining | if put is live: put all remaining 25%
State 20% | partial calls 5%-13% in 1% steps -> leave 7%-15% remaining; call 20% -> 0% remaining | if put is live: put all remaining 20%
State 15% | partial calls 5%-8% in 1% steps -> leave 7%-10% remaining; call 15% -> 0% remaining | if put is live: put all remaining 15%
State 10% | call 10% -> 0% remaining | if put is live: put all remaining 10%
St

## 7. DLOM and Primary Fair Value

Longstaff's lookback-put approach is retained as the marketability adjustment.
The primary reported value is the integrated clean contract value after applying the central DLOM estimate.

In [10]:
def longstaff_dlom(paths, rf, dt, spot0, restriction_years):
    idx = min(int(round(restriction_years / dt)), paths.shape[1] - 1)
    actual_horizon = idx * dt
    s_max = paths[:, : idx + 1].max(axis=1)
    s_T = paths[:, idx]
    lookback_payoff = s_max - s_T
    lookback_value = np.exp(-rf * actual_horizon) * lookback_payoff.mean()
    return {
        "restriction_years": actual_horizon,
        "lookback_value": lookback_value,
        "dlom_pct": lookback_value / spot0,
    }


dlom_terms = {T: longstaff_dlom(paths, risk_free_rate, dt, SPA_price, T) for T in [3.0, 4.0, 5.0]}
dlom_central = dlom_terms[4.0]["dlom_pct"]
full_value_total_dlom = full_value_total * (1 - dlom_central)
full_value_per_share_dlom = full_value_per_share * (1 - dlom_central)

print("DLOM via Longstaff lookback put")
print("-" * 72)
for T in [3.0, 4.0, 5.0]:
    term = dlom_terms[T]
    print(
        f"Restriction horizon {term['restriction_years']:.1f}yr | "
        f"Lookback value {term['lookback_value']:>7.3f} | "
        f"DLOM {term['dlom_pct']:.2%}"
    )
print("-" * 72)
print(f"Primary fair value after central DLOM: USD {full_value_total_dlom / 1e6:,.1f}M")
print(f"Primary fair value per option share:   USD {full_value_per_share_dlom:,.2f}")

DLOM via Longstaff lookback put
------------------------------------------------------------------------
Restriction horizon 3.0yr | Lookback value 107.947 | DLOM 18.70%
Restriction horizon 4.0yr | Lookback value 121.699 | DLOM 21.08%
Restriction horizon 5.0yr | Lookback value 132.735 | DLOM 23.00%
------------------------------------------------------------------------
Primary fair value after central DLOM: USD 111.2M
Primary fair value per option share:   USD 80.24


## 8. Precision and Sensitivities

All reported scenario runs below use 1,000,000 simulations.
The notebook reports observed precision at that fixed simulation policy and then shows market and contract sensitivities on the same path count. It does not present a separate simulation-count sensitivity table.

In [11]:
print("Numerical precision and runtime controls")
print("-" * 72)
print(f"Core vGameTheory valuation uses a deterministic tree with {tree_steps} monthly steps.")
print(f"Live remaining-state levels on the grid:   {len(STATE_PCTS) - 1}")
print("Monte Carlo standard error on pre-DLOM contract value: not applicable")
print(f"MC paths retained for DLOM:               {simulations:,}")
print(f"Slow sensitivity cells enabled:           {'Yes' if RUN_SLOW_SENSITIVITIES else 'No - skipped by default'}")

Numerical precision and runtime controls
------------------------------------------------------------------------
Core vGameTheory valuation uses a deterministic tree with 72 monthly steps.
Live remaining-state levels on the grid:   19
Monte Carlo standard error on pre-DLOM contract value: not applicable
MC paths retained for DLOM:               1,000,000
Slow sensitivity cells enabled:           No - skipped by default


In [12]:
if RUN_SLOW_SENSITIVITIES:
    vol_range = [0.15, 0.20, 0.25, 0.30]
    rf_range = [0.04, risk_free_rate, 0.06]

    print("vGameTheory fair value to Olympus BV (USD millions, before DLOM)")
    print(f"{'Asset Vol':>10} | " + " | ".join(f" rf={rf:.2%} " for rf in rf_range))
    print("-" * (14 + 15 * len(rf_range)))

    base_value_m = full_value_total / 1e6
    for vol in vol_range:
        row = []
        for rf in rf_range:
            is_base_case = vol == asset_volatility and abs(rf - risk_free_rate) < 1e-12
            if is_base_case:
                value_m = base_value_m
            else:
                result = run_vgame_model_scenario(
                    vol=vol,
                    rf=rf,
                    allow_partial_calls=ALLOW_PARTIAL_CALLS,
                    tie_priority_to_olympus=GAME_TIE_PRIORITY_TO_OLYMPUS,
                )
                value_m = result["value_total"] / 1e6
            marker = " *" if is_base_case else "  "
            row.append(f"{value_m:>8.1f}{marker}")
        print(f"{vol:>9.0%} | " + " | ".join(row))

    print("\nBase case marked with *.")
else:
    print("Slow market sensitivity grid skipped.")
    print("Set RUN_SLOW_SENSITIVITIES = True in Cell 4 of this notebook to execute it.")

Slow market sensitivity grid skipped.
Set RUN_SLOW_SENSITIVITIES = True in Cell 4 of this notebook to execute it.


## 9. Contract-Specific Sensitivities and Scope

The remaining uncertainties in the FULL build are not generic option inputs; they are contract-specific modeling choices.
This section keeps the simulation count fixed at 1,000,000 and changes only those contract-specific assumptions while keeping regulatory lapse risk qualitative only.

In [13]:
if RUN_SLOW_SENSITIVITIES:
    structural_cases = [
        ("Base case: partial calls, 50/50 tie priority", True, 0.50),
        ("Full-block only, 50/50 tie priority", False, 0.50),
        ("Partial calls, Olympus 25% tie priority", True, 0.25),
        ("Partial calls, Olympus 75% tie priority", True, 0.75),
    ]

    print("Structural sensitivity under vGameTheory (USD millions, before DLOM)")
    print("-" * 72)
    for label, allow_partial, tie_priority in structural_cases:
        is_base_case = allow_partial == ALLOW_PARTIAL_CALLS and abs(tie_priority - GAME_TIE_PRIORITY_TO_OLYMPUS) < 1e-12
        if is_base_case:
            value_total = full_value_total
        else:
            result = run_vgame_model_scenario(
                vol=asset_volatility,
                rf=risk_free_rate,
                allow_partial_calls=allow_partial,
                tie_priority_to_olympus=tie_priority,
            )
            value_total = result["value_total"]
        print(f"{label:<48} {value_total / 1e6:>8.1f}M")

    print("\nRegulatory lapse risk remains qualitative and is not deducted from the primary value.")
else:
    print("Slow structural sensitivity grid skipped.")
    print("Set RUN_SLOW_SENSITIVITIES = True in Cell 4 of this notebook to execute it.")

Slow structural sensitivity grid skipped.
Set RUN_SLOW_SENSITIVITIES = True in Cell 4 of this notebook to execute it.


In [14]:
print("Contract-feasible call-size ranges under the vGameTheory grid")
print("-" * 72)
for current_pct in [25, 20, 15, 10]:
    print(f"From {current_pct:>2}% remaining: {summarize_call_menu(current_pct, ALLOW_PARTIAL_CALLS)}")

print("\nTitan's put rule: at any eligible put date, Titan may put all remaining shares in the current state only.")

Contract-feasible call-size ranges under the vGameTheory grid
------------------------------------------------------------------------
From 25% remaining: partial calls 5%-18% in 1% steps -> leave 7%-20% remaining; call 25% -> 0% remaining
From 20% remaining: partial calls 5%-13% in 1% steps -> leave 7%-15% remaining; call 20% -> 0% remaining
From 15% remaining: partial calls 5%-8% in 1% steps -> leave 7%-10% remaining; call 15% -> 0% remaining
From 10% remaining: call 10% -> 0% remaining

Titan's put rule: at any eligible put date, Titan may put all remaining shares in the current state only.


In [15]:
titan_value_total_dlom = -full_value_total_dlom
titan_value_per_share_dlom = -full_value_per_share_dlom

print("=" * 72)
print("Primary reported output - vGameTheory")
print("=" * 72)
print(f"Value to Olympus before DLOM:     USD {full_value_total / 1e6:,.1f}M")
print(f"Mirror value to Titan:            USD {titan_value_total / 1e6:,.1f}M")
print(f"Value to Olympus after DLOM:      USD {full_value_total_dlom / 1e6:,.1f}M")
print(f"Mirror value to Titan after DLOM: USD {titan_value_total_dlom / 1e6:,.1f}M")
print(f"Per option share to Olympus:      USD {full_value_per_share_dlom:,.2f}")
print(f"Per option share to Titan:        USD {titan_value_per_share_dlom:,.2f}")
print()
print("Scope note:")
print("- overlap is modeled as a discrete-time zero-sum game approximation")
print("- slow sensitivity cells are skipped unless RUN_SLOW_SENSITIVITIES is set to True")
print("- regulatory lapse risk remains qualitative only and is not deducted from the primary value")
print("- leakage, dividends, and SPA claim adjustments remain outside the primary model")
if pending_source_items:
    print("- source-control exception(s) remain outstanding before external issue:")
    for name in pending_source_items:
        print(f"  * {name}")
else:
    print("- source-control register complete")

Primary reported output - vGameTheory
Value to Olympus before DLOM:     USD 140.9M
Mirror value to Titan:            USD -140.9M
Value to Olympus after DLOM:      USD 111.2M
Mirror value to Titan after DLOM: USD -111.2M
Per option share to Olympus:      USD 80.24
Per option share to Titan:        USD -80.24

Scope note:
- overlap is modeled as a discrete-time zero-sum game approximation
- slow sensitivity cells are skipped unless RUN_SLOW_SENSITIVITIES is set to True
- regulatory lapse risk remains qualitative only and is not deducted from the primary value
- leakage, dividends, and SPA claim adjustments remain outside the primary model
- source-control exception(s) remain outstanding before external issue:
  * TOTAL_EQUITY_USD / SPA_price
